In [ ]:
import json

In [ ]:
class Pessoa:
    def __init__(self, evento: dict):
        self.last_timestamp = evento["timestamp"]
        self.last_zone = evento["zone_id"]
        self.last_event = evento["event_type"]
        self.genero = evento["gender"]
        self.idade = evento["age_range"]

    def calcular_score_match(self, novo_evento):
        """
        Retorna um valor de 0 a 100 (ou True/False) indicando 
        a probabilidade deste evento ser desta pessoa.
        """
        # 1. Diferença de Tempo (Gap < 300s)
        delta_t = (novo_evento["timestamp"] - self.last_timestamp).total_seconds()
        if delta_t < 0 or delta_t > 300: # Se o evento é no passado ou passou 5 min
            return 0
        
        # 2. Plausibilidade Espacial (Usando o teu Grafo)
        # Verifica se existe ligação entre a última zona e a nova
        distancia_minima = 0
        if self.last_zone != novo_evento["zone_id"]:
            if not grafo.has_edge(self.last_zone, novo_evento["zone_id"]):
                # Se não houver ligação direta, podemos ser tolerantes 
                # mas penalizar o score
                return 10 
            distancia_minima = grafo[self.last_zone][novo_evento["zone_id"]]['weight']
            
            # Se a pessoa chegou mais rápido do que o "walk_seconds", é suspeito
            if delta_t < distancia_minima:
                return 20

        # 3. Consistência de Atributos (Margem de erro 8% e 12%)
        score_atributos = 0
        if self.genero == novo_evento["gender"]:
            score_atributos += 40
        if self.idade == novo_evento["age_range"]:
            score_atributos += 40
            
        return score_atributos + 20 # Base de confiança se passou no tempo e espaço

In [14]:
with open("zones.json", "r", encoding="utf-8") as f:
    info_zonas: dict[str, dict] = json.load(f)

loja = info_zonas["zones"]

for zone_name in loja.keys():
    loja[zone_name]["pessoas"]: dict[int, Pessoa] = dict() # type: ignore

In [ ]:
import pandas as pd

df_eventos = pd.read_csv("events.csv", dtype={
    "event_id": "str",
    "timestamp": "str",
    "zone_id": "category",
    "event_type": "category",
    "duration_s": "int32",
    "gender": "category",
    "age_range": "category"
})

contador_id = 1

ids_atribuidos = []

zonas_ativas: dict[str, dict] = dict()
"""
Lista de Nodes que têm la alguem

key: string (id_zona)
item: dict (obj_zona)

Isto vai servir para acessar fácilmente para tirar de la pessoas,
 cujas, ATÉ entrarem noutra zona com sucesso, ficam aqui.

"""



for evento in df_eventos.itertuples(index=True, name="Evento"):
    tempo_atual = evento.timestamp
    zona_atual = loja[evento.zone_id]
    pessoa_evento = Pessoa(evento)
    pessoa_atual = None

    # Entrada em Zona
    match evento.event_type:
        case "entry":

    #       Nova pessoa entrou na loja
            if evento.zone_id in {"Z_E1", "Z_E2"}:
                id_pessoa_atual = contador_id
                contador_id += 1

                # Adicionar Pessoa à respetiva zona
                zona_atual["pessoas"][id_pessoa_atual] = pessoa_evento
                zonas_ativas[evento.zone_id] = zona_atual
                
                continue
    

    #       Alguem entrou noutra zona

            # Ver zonas possíveis
            zonas_possíveis = zona_atual["adjacent"]

            # Iterar pelas zonas adjacentes à procura da pessoa
            for nome_zona in zonas_possíveis:
                
                #Zona não tem ninguém, skip
                if nome_zona not in zonas_ativas:
                    continue

                # Procurar por pessoas em zonas ativas
                for pessoa in loja[nome_zona]["pessoas"]:
                    if pessoa.last_event == "exit":
                        pass



    # Fim

    ids_atribuidos.append(pessoa_atual)
            

    


IndentationError: expected an indented block after 'case' statement on line 47 (3161308988.py, line 49)

In [ ]:
import matplotlib.pyplot as plt

def visualizar_loja(G):
    plt.figure(figsize=(12, 8))
    
    # 1. Definir o layout (spring_layout ajuda a separar os nós)
    pos = nx.spring_layout(G, seed=42) 
    
    # 2. Desenhar os nós e as ligações
    nx.draw(G, pos, with_labels=True, 
            node_color="lightblue", 
            node_size=1500, 
            font_size=8, 
            font_weight="bold",
            edge_color="gray")
    
    # 3. Desenhar os pesos (walk_seconds) nas arestas
    edge_labels = nx.get_edge_attributes(G, "weight")
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)
    
    plt.title("Topologia da Loja - Grafo de Transições (Segundos)")
    plt.show()

# Chamar a função depois de populares o grafo
visualizar_loja(loja)